# Qwen continuation dataset generator

Run with a GPU runtime.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print(torch.cuda.get_device_name(0))


In [ ]:
import os

try:
    import google.colab
    ENV = 'colab'
    BASE_DIR = '/content'
except ImportError:
    if os.path.exists('/kaggle'):
        ENV = 'kaggle'
        BASE_DIR = '/kaggle/working'
    else:
        ENV = 'local'
        BASE_DIR = os.getcwd()

print(f'Environment: {ENV}, base dir: {BASE_DIR}')


In [ ]:
REPO_URL = 'https://github.com/Teodorus730/qwen.git'
BRANCH = 'main'
PROJECT_DIR = 'qwen_continuation_dataset'
REPO_DIR = f'{BASE_DIR}/qwen'

!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}/{PROJECT_DIR}
!python -m pip install -q -r requirements.txt


## Validate configuration

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --output outputs/mixed_entropy_20.jsonl \
  --overwrite \
  --dry-run


## Generate — fixed mode (FineWeb-Edu)

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset fineweb \
  --mode fixed \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/fineweb_fixed_20.jsonl \
  --overwrite


## Generate — entropy mode (FineMath)

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset math \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/math_entropy_20.jsonl \
  --overwrite


## Generate — mixed sources

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.7 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/mixed_70_entropy_20.jsonl \
  --overwrite


## Inspect results

In [ ]:
!python inspect_dataset.py \
    outputs/mixed_70_entropy_20.jsonl \
    --show-examples 3


## Cycle detection off (comparison)

Same settings as the fixed-mode example above, but with n-gram cycle
detection explicitly disabled — useful for comparing raw model behavior
against the cycle-checked output.

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset fineweb \
  --mode fixed \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --no-cycle-detection \
  --max-examples 20 \
  --preview 3 \
  --output outputs/fineweb_fixed_no_cycle.jsonl \
  --overwrite


## Hugging Face upload

No need to edit `config.yaml` — pass `--hf-upload` and `--hf-repo-id`
directly. Authorise below, then run. Shards upload incrementally and
resume from the last checkpoint on restart, so this cell is safe to
re-run after a disconnect.

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste token or run: !huggingface-cli login
HF_REPO_ID = 'your_username/qwen_continuation_dataset'


In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 50000 \
  --output outputs/train.jsonl \
  --hf-upload \
  --hf-repo-id $HF_REPO_ID \
  --hf-shard-size 10000
